In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:

import os
#使用製作好的離線安裝包
PACKAGE_DIR = '/kaggle/input/xlstm-package-v2' 

#執行離線安裝
!pip install xlstm ninja triton --no-index --find-links=$PACKAGE_DIR

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from sklearn.preprocessing import LabelEncoder, StandardScaler
from torch.utils.data import Subset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error
import xgboost as xgb
from tqdm.auto import tqdm
import warnings
import os
import gc
from torch.utils.data import Subset
import random
from sklearn.model_selection import KFold, train_test_split
warnings.filterwarnings('ignore')

#設定隨機種子以確保結果可重現
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)

class Config:
    MAX_SEQ_LEN = 4096
    BATCH_SIZE = 2       
    GRAD_ACCUM_STEPS = 16
    EMBED_DIM = 64
    NUM_BLOCKS = 2 
    LEARNING_RATE = 3e-4
    EPOCHS = 25
    WEIGHT_DECAY = 0.1
    
    BASE_DIR = '/kaggle/input/linking-writing-processes-to-writing-quality'
    TRAIN_LOGS_PATH = f'{BASE_DIR}/train_logs.csv'
    TRAIN_SCORES_PATH = f'{BASE_DIR}/train_scores.csv'
    TEST_LOGS_PATH = f'{BASE_DIR}/test_logs.csv'
    
    MODEL_SAVE_PATH = "xlstm_model.pth"

cfg = Config()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用CPU/GPU: {device}")

try:
    from xlstm import xLSTMBlockStack, xLSTMBlockStackConfig, mLSTMBlockConfig, mLSTMLayerConfig, sLSTMBlockConfig, sLSTMLayerConfig, FeedForwardConfig
    print("成功載入xlstm")
except ImportError:
    raise ImportError("xLSTM載入失敗")

In [ ]:

#資料預處理，文件3.2部分
def preprocess_logs(logs):
    
    logs['prev_up_time'] = logs.groupby('id')['up_time'].shift(1)
    logs['iki_raw'] = logs['down_time'] - logs['prev_up_time']
    logs['iki_raw'] = logs['iki_raw'].fillna(0).clip(lower=0)
    logs['iki'] = np.log1p(logs['iki_raw'])
    
    logs['word_count_delta'] = logs.groupby('id')['word_count'].diff().fillna(0)

    logs['is_pause'] = (logs['iki_raw'] > 2000).astype(int)
    
    logs['burst_group'] = logs.groupby('id')['is_pause'].cumsum()

    logs['time_since_last_pause'] = logs.groupby(['id', 'burst_group'])['action_time'].cumsum()

    logs['time_since_last_pause'] = np.log1p(logs['time_since_last_pause'])


    logs['prev_cursor'] = logs.groupby('id')['cursor_position'].shift(1).fillna(0)
    logs['cursor_jump_distance'] = (logs['cursor_position'] - logs['prev_cursor']).abs()
    logs['cursor_jump_distance'] = np.log1p(logs['cursor_jump_distance'])

    
    logs['prev_activity'] = logs.groupby('id')['activity'].shift(1)
    
    logs['is_typo_correction'] = (
        (logs['down_event'] == 'Backspace') & 
        (logs['prev_activity'] == 'Input') & 
        (logs['iki_raw'] < 2000)
    ).astype(int)

    return logs

def encode_and_scale(logs, encoders=None, scaler=None):

    num_cols = [
        'action_time', 'cursor_position', 'word_count', 'word_count_delta', 'iki',
        'time_since_last_pause', 'cursor_jump_distance', 'is_typo_correction'
    ]
    
    if encoders is None:
        encoders = {}
        is_train = True
    else:
        is_train = False

    for col in ['activity', 'down_event', 'up_event']:
        logs[col] = logs[col].astype(str)
        if is_train:
            le = LabelEncoder()
            logs[col] = le.fit_transform(logs[col])
            encoders[col] = le
        else:
            le = encoders[col]
            logs[col] = logs[col].apply(lambda x: le.transform([x])[0] if x in le.classes_ else 0)

    logs[num_cols] = logs[num_cols].replace([np.inf, -np.inf], 0).fillna(0)

    if is_train:
        scaler = StandardScaler()
        logs[num_cols] = scaler.fit_transform(logs[num_cols])
    else:
        logs[num_cols] = scaler.transform(logs[num_cols])

    n_act = len(encoders['activity'].classes_)
    n_down = len(encoders['down_event'].classes_)
    n_up = len(encoders['up_event'].classes_)
    
    return logs, encoders, scaler, n_act, n_down, n_up


def prepare_data(logs_path, scores_path=None, encoders=None, scaler=None):
    print(f"從{logs_path}載入資料")
    logs = pd.read_csv(logs_path)

    logs = preprocess_logs(logs)

    logs, encoders, scaler, n_act, n_down, n_up = encode_and_scale(logs, encoders, scaler)
    
    scores_map = {}
    if scores_path and os.path.exists(scores_path):
        scores_df = pd.read_csv(scores_path)
        scores_map = dict(zip(scores_df['id'], scores_df['score']))
        
    return logs, scores_map, encoders, scaler, n_act, n_down, n_up

In [ ]:
import re
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer

#手工統計特徵helper function,文件3.3部分

#近似重構
def reconstruct_essay(essay_df):

    events = essay_df[['activity', 'cursor_position', 'text_change']].to_dict('records')
    
    text_buffer = []
    
    for event in events:
        act = event['activity']
        txt = str(event['text_change'])
        
        if act == 'Input':
            if txt != 'NoChange':
                text_buffer.append(txt)
        elif act in ['Remove/Cut']:
            if text_buffer:
                text_buffer.pop()
        elif act == 'Replace':
            if text_buffer:
                text_buffer.pop()
                text_buffer.append(txt)
                
    full_text = "".join(text_buffer)
    return full_text

#文章結構特徵
def get_structural_features(text):
    features = {}
    
    paragraphs = [p for p in text.split('\n') if len(p) > 0]
    features['para_count'] = len(paragraphs)
    
    if len(paragraphs) > 0:
        para_lens = [len(p) for p in paragraphs]
        features['para_len_mean'] = np.mean(para_lens)
        features['para_len_std'] = np.std(para_lens)
        features['para_len_max'] = np.max(para_lens)

        for i in range(3):
            features[f'para_len_{i+1}'] = para_lens[i] if i < len(paragraphs) else 0
    else:
        features['para_len_mean'] = 0
        features['para_len_std'] = 0
        features['para_len_max'] = 0
        for i in range(3): features[f'para_len_{i+1}'] = 0

    sentences = re.split(r'[.?!]+', text)
    sentences = [s for s in sentences if len(s.strip()) > 0]
    features['sent_count'] = len(sentences)
    
    if len(sentences) > 0:
        sent_lens = [len(s) for s in sentences]
        features['sent_len_mean'] = np.mean(sent_lens)
        features['sent_len_std'] = np.std(sent_lens)
        features['sent_len_max'] = np.max(sent_lens)

        for i in range(3):
            features[f'sent_len_{i+1}'] = sent_lens[i] if i < len(sentences) else 0
    else:
        features['sent_len_mean'] = 0
        features['sent_len_std'] = 0
        features['sent_len_max'] = 0
        for i in range(3): features[f'sent_len_{i+1}'] = 0
        
    features['punct_commas'] = text.count(',')
    features['punct_periods'] = text.count('.')
    features['punct_questions'] = text.count('?')
    features['punct_exclamations'] = text.count('!')
    features['punct_total'] = features['punct_commas'] + features['punct_periods'] + features['punct_questions'] + features['punct_exclamations']

    return features

#寫作過程特徵
def get_process_features(df):

    features = {}

    timestamps = [7, 15, 22, 35]
    for m in timestamps:
        threshold = m * 60 * 1000
        temp_df = df[df['down_time'] < threshold]
        if len(temp_df) > 0:
            features[f'word_count_at_{m}m'] = temp_df['word_count'].max()
            features[f'action_count_at_{m}m'] = len(temp_df)
        else:
            features[f'word_count_at_{m}m'] = 0
            features[f'action_count_at_{m}m'] = 0
            
    features['cursor_std_log'] = np.log1p(df['cursor_position'].std())

    df['prev_cursor'] = df['cursor_position'].shift(1).fillna(0)
    df['cursor_jump'] = df['cursor_position'] - df['prev_cursor']
    features['non_sequential_count'] = ((df['cursor_jump'].abs() > 1) & (df['activity'] == 'Input')).sum()
    
    features['total_shift_usage'] = df['down_event'].apply(lambda x: 'Shift' in str(x)).sum()
    
    features['replace_count'] = (df['activity'] == 'Replace').sum()
    
    return features

#停頓特徵
def get_pause_features(df):

    feats = {}
    
    raw_iki = np.expm1(df['iki']).fillna(0) 

    n_actions = len(df)
    feats['pause_long_count'] = (raw_iki >= 2000).sum()
    feats['pause_mid_count'] = ((raw_iki > 1000) & (raw_iki <= 1999)).sum()
    feats['pause_short_count'] = ((raw_iki >= 300) & (raw_iki <= 999)).sum()

    feats['pause_long_ratio'] = feats['pause_long_count'] / (n_actions + 1)
    feats['pause_mid_ratio'] = feats['pause_mid_count'] / (n_actions + 1)
    feats['pause_short_ratio'] = feats['pause_short_count'] / (n_actions + 1)

    iki_mean = raw_iki.mean()
    iki_std = raw_iki.std()
    feats['iki_cv'] = iki_std / (iki_mean + 1e-6)
    
    return feats

#修改行為特徵
def get_editing_features(df, text_len):
    """
    優化後的修改行為特徵
    """
    feats = {}
    n_actions = len(df)
    
    feats['productivity_ratio'] = text_len / (n_actions + 1)

    corrections = df['down_event'].isin(['Backspace', 'Delete']).sum()
    feats['correction_ratio'] = corrections / (n_actions + 1)
    
    total_time_min = (df['down_time'].max() - df['down_time'].min()) / 60000
    if total_time_min > 0:
        feats['chars_per_min'] = text_len / total_time_min
    else:
        feats['chars_per_min'] = 0
        
    return feats

#寫作過程前半/後半比較
def get_segmented_features(df):

    feats = {}

    max_time = df['down_time'].max()
    mid_time = max_time / 2

    first_half = df[df['down_time'] <= mid_time]
    second_half = df[df['down_time'] > mid_time]

    prod_rate_1 = len(first_half[first_half['activity'] == 'Input']) / (len(first_half) + 1)
    prod_rate_2 = len(second_half[second_half['activity'] == 'Input']) / (len(second_half) + 1)
    
    feats['prod_rate_first_half'] = prod_rate_1
    feats['prod_rate_second_half'] = prod_rate_2
    feats['prod_rate_change'] = prod_rate_2 - prod_rate_1

    iki_1 = first_half['iki'].mean() if len(first_half) > 0 else 0
    iki_2 = second_half['iki'].mean() if len(second_half) > 0 else 0
    
    feats['iki_mean_first_half'] = iki_1
    feats['iki_mean_second_half'] = iki_2
    feats['iki_change_ratio'] = iki_2 / (iki_1 + 1e-6)

    corr_1 = first_half['down_event'].isin(['Backspace', 'Delete']).sum()
    corr_2 = second_half['down_event'].isin(['Backspace', 'Delete']).sum()
    
    feats['corr_count_first'] = corr_1
    feats['corr_count_second'] = corr_2
    feats['corr_ratio_change'] = (corr_2 + 1) / (corr_1 + 1)
    
    return feats

In [ ]:
#執行製作手工特徵
def make_handcrafted_features(logs, tfidf_vectorizer=None, is_train=True):
    print("正在生成化手工特徵")
    ids = logs['id'].unique()
    
    print("正在準備事件序列")
    logs['down_event_str'] = logs['down_event'].astype(str)
    
    grp = logs.groupby('id')
    basic_feats = pd.DataFrame({'id': ids})

    time_stats = grp['action_time'].agg(['mean', 'max', 'std', 'sum']).add_prefix('action_time_')
    iki_stats = grp['iki'].agg(['mean', 'max', 'std', 'sum', 'median']).add_prefix('iki_')
    word_stats = grp['word_count'].agg(['max', 'last']).add_prefix('word_count_')
    cursor_stats = grp['cursor_position'].agg(['std', 'max']).add_prefix('cursor_')
    
    basic_feats = basic_feats.merge(time_stats, on='id', how='left')
    basic_feats = basic_feats.merge(iki_stats, on='id', how='left')
    basic_feats = basic_feats.merge(word_stats, on='id', how='left')
    basic_feats = basic_feats.merge(cursor_stats, on='id', how='left')

    activity_stats = grp['activity'].value_counts().unstack(fill_value=0)
    activity_stats.columns = [f'act_cnt_{i}' for i in activity_stats.columns]
    basic_feats = basic_feats.merge(activity_stats, on='id', how='left')

    print("正在提取結構、過程、停頓與編輯特徵")
    advanced_feats_list = []
    
    #新增各手工特徵
    for essay_id, essay_df in tqdm(grp, total=len(ids)):
        row_feats = {'id': essay_id}

        text = reconstruct_essay(essay_df)
        struct_feats = get_structural_features(text)
        row_feats.update(struct_feats)

        process_feats = get_process_features(essay_df)
        row_feats.update(process_feats)

        pause_feats = get_pause_features(essay_df)
        row_feats.update(pause_feats)
        
        edit_feats = get_editing_features(essay_df, len(text))
        row_feats.update(edit_feats)

        segmented_feats=get_segmented_features(essay_df)
        row_feats.update(segmented_feats)
        
        advanced_feats_list.append(row_feats)
        
    advanced_df = pd.DataFrame(advanced_feats_list)
    
    full_feats = basic_feats.merge(advanced_df, on='id', how='left')
    
    #TF-IDF
    corpus_df = logs.groupby('id')['down_event_str'].apply(lambda x: " ".join(x)).reset_index()
    corpus = corpus_df.set_index('id').reindex(ids)['down_event_str'].fillna("")
    
    if is_train:
        if tfidf_vectorizer is None:
            tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 3), max_features=100)
        tfidf_mat = tfidf_vectorizer.fit_transform(corpus)
    else:
        tfidf_mat = tfidf_vectorizer.transform(corpus)
        
    tfidf_df = pd.DataFrame(tfidf_mat.toarray(), columns=[f'evt_tfidf_{i}' for i in range(tfidf_mat.shape[1])])
    tfidf_df['id'] = ids
    
    full_feats = full_feats.merge(tfidf_df, on='id', how='left')
    
    return full_feats, tfidf_vectorizer

In [ ]:
#準備 xlstm dataset，與建立xlstm的模型

class KeystrokeDataset(Dataset):
    def __init__(self, logs_df, scores_map=None, max_len=4096):
        self.logs = logs_df
        self.scores_map = scores_map
        self.max_len = max_len
        self.essay_ids = logs_df['id'].unique()
        self.grouped = logs_df.groupby('id')
        
    def __len__(self):
        return len(self.essay_ids)
    
    def __getitem__(self, idx):
        essay_id = self.essay_ids[idx]
        df = self.grouped.get_group(essay_id)

        feat_cat = df[['activity', 'down_event', 'up_event']].values

        feat_num = df[[
            'action_time', 
            'cursor_position', 
            'word_count', 
            'word_count_delta', 
            'iki',
            'time_since_last_pause',
            'cursor_jump_distance',
            'is_typo_correction'
        ]].values

        curr_len = len(df)
        padded_cat = np.zeros((self.max_len, 3), dtype=np.int64)

        padded_num = np.zeros((self.max_len, 8), dtype=np.float32) 
        # =======================================
        
        real_len = min(curr_len, self.max_len)
        padded_cat[:real_len] = feat_cat[:real_len]
        padded_num[:real_len] = feat_num[:real_len]

        score = self.scores_map.get(essay_id, 0.0) if self.scores_map else 0.0
        
        return {
            'cat_features': torch.tensor(padded_cat),
            'num_features': torch.tensor(padded_num),
            'label': torch.tensor(score, dtype=torch.float32)
        }

#建立Attention Pooling方法
class AttentionPooling(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(in_dim, in_dim),
            nn.Tanh(),
            nn.Linear(in_dim, 1, bias=False)
        )
        
    def forward(self, x):
        w = self.attention(x)
        w = torch.softmax(w, dim=1)
        x = torch.sum(x * w, dim=1)
        return x

class xLSTMModel(nn.Module):
    def __init__(self, n_act, n_down, n_up, embed_dim=64): 
        super().__init__()

        self.act_emb = nn.Embedding(n_act + 1, 16)
        self.down_emb = nn.Embedding(n_down + 1, 32)
        self.up_emb = nn.Embedding(n_up + 1, 32)

        self.input_proj = nn.Linear(16 + 32 + 32 + 8, embed_dim)

        xlstm_cfg = xLSTMBlockStackConfig(
            mlstm_block=mLSTMBlockConfig(
                mlstm=mLSTMLayerConfig(conv1d_kernel_size=4, qkv_proj_blocksize=4, num_heads=4)
            ),
            slstm_block=sLSTMBlockConfig(
                slstm=sLSTMLayerConfig(backend="cuda", num_heads=4, conv1d_kernel_size=4, bias_init="powerlaw_blockdependent"),
                feedforward=FeedForwardConfig(proj_factor=1.3, act_fn="gelu"),
            ),
            context_length=cfg.MAX_SEQ_LEN,
            num_blocks=cfg.NUM_BLOCKS, 
            embedding_dim=embed_dim,
            slstm_at=[1] 
        )
        self.xlstm = xLSTMBlockStack(xlstm_cfg)

        self.attn_pooling = AttentionPooling(embed_dim)

        self.regressor = nn.Sequential(
            nn.Linear(embed_dim * 2, 64), 
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )
        
    def forward(self, cat_feats, num_feats):

        e1 = self.act_emb(cat_feats[:, :, 0])
        e2 = self.down_emb(cat_feats[:, :, 1])
        e3 = self.up_emb(cat_feats[:, :, 2])
        
        x_seq = torch.cat([e1, e2, e3, num_feats], dim=-1)

        x_seq = self.input_proj(x_seq)
        x_seq = self.xlstm(x_seq)

        pool_attn = self.attn_pooling(x_seq)
        pool_mean = torch.mean(x_seq, dim=1)
        
        x_final = torch.cat([pool_attn, pool_mean], dim=1)

        return self.regressor(x_final)

    #只提取權重中特徵(inference用)        
    def extract_features(self, cat_feats, num_feats):

        e1 = self.act_emb(cat_feats[:, :, 0])
        e2 = self.down_emb(cat_feats[:, :, 1])
        e3 = self.up_emb(cat_feats[:, :, 2])

        x_seq = torch.cat([e1, e2, e3, num_feats], dim=-1)

        x_seq = self.input_proj(x_seq)
        x_seq = self.xlstm(x_seq)

        pool_attn = self.attn_pooling(x_seq)
        pool_mean = torch.mean(x_seq, dim=1)
        x_final = torch.cat([pool_attn, pool_mean], dim=1)
        
        return x_final

In [ ]:
#載入並處理 Train Data
print(f"載入訓練資料({cfg.TRAIN_LOGS_PATH})")
train_logs = pd.read_csv(cfg.TRAIN_LOGS_PATH)
train_logs = preprocess_logs(train_logs)
train_logs, encoders, scaler, n_act, n_down, n_up = encode_and_scale(train_logs, None, None)

scores_df = pd.read_csv(cfg.TRAIN_SCORES_PATH)
scores_map = dict(zip(scores_df['id'], scores_df['score']))

train_handcrafted, tfidf_vec = make_handcrafted_features(train_logs, is_train=True)
train_handcrafted = train_handcrafted.merge(scores_df, on='id', how='left')

dataset = KeystrokeDataset(train_logs, scores_map, max_len=cfg.MAX_SEQ_LEN)
dataloader = DataLoader(dataset, batch_size=cfg.BATCH_SIZE, shuffle=True, num_workers=0)

train_idx, val_idx = train_test_split(np.arange(len(dataset)), test_size=0.2, random_state=42)
train_loader = DataLoader(Subset(dataset, train_idx), batch_size=cfg.BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(Subset(dataset, val_idx), batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=0)

#初始化模型
model = xLSTMModel(n_act, n_down, n_up, embed_dim=cfg.EMBED_DIM).to(device)
criterion = nn.MSELoss()
optimizer = optim.AdamW(model.parameters(), lr=cfg.LEARNING_RATE, weight_decay=cfg.WEIGHT_DECAY)
grad_scaler = GradScaler()
total_steps = (len(dataloader) // cfg.GRAD_ACCUM_STEPS) * cfg.EPOCHS
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)

best_val_rmse = float('inf')
history = {'train_rmse': [], 'val_rmse': []}

In [ ]:
#訓練 xLSTM
print("\n開始訓練")
for epoch in range(cfg.EPOCHS):
    model.train()
    train_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Ep {epoch+1}/{cfg.EPOCHS}", leave=False)
        
    for batch_idx, batch in enumerate(progress_bar):
        cat = batch['cat_features'].to(device)
        num = batch['num_features'].to(device)
        
        labels = batch['label'].to(device)
            
        with autocast(device_type='cuda', dtype=torch.float16):
            
            outputs = model(cat, num).view(-1)
            loss = criterion(outputs, labels) / cfg.GRAD_ACCUM_STEPS
            
        grad_scaler.scale(loss).backward()
        train_loss += loss.item() * cfg.GRAD_ACCUM_STEPS
            
        if (batch_idx + 1) % cfg.GRAD_ACCUM_STEPS == 0:
            grad_scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            grad_scaler.step(optimizer)
            grad_scaler.update()
            scheduler.step()
            optimizer.zero_grad()
        
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            cat = batch['cat_features'].to(device)
            num = batch['num_features'].to(device)
            labels = batch['label'].to(device)
                
            outputs = model(cat, num).view(-1)
            outputs = torch.clamp(outputs, 0.5, 6.0)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
        
    avg_train_rmse = np.sqrt(train_loss / len(train_loader))
    avg_val_rmse = np.sqrt(val_loss / len(val_loader))
        
    history['train_rmse'].append(avg_train_rmse)
    history['val_rmse'].append(avg_val_rmse)
        
    if avg_val_rmse < best_val_rmse:
        best_val_rmse = avg_val_rmse
        torch.save(model.state_dict(), cfg.MODEL_SAVE_PATH)
        print(f"Epoch {epoch+1}: Train RMSE: {avg_train_rmse:.4f} | New Best Val RMSE: {best_val_rmse:.4f} (Saved!)")
    else:
        print(f"Epoch {epoch+1}: Train RMSE: {avg_train_rmse:.4f} | Val RMSE: {avg_val_rmse:.4f}")
    
#訓練曲線圖
plt.figure(figsize=(10, 6))
plt.plot(range(1, cfg.EPOCHS + 1), history['train_rmse'], label='Training RMSE', marker='o')
plt.plot(range(1, cfg.EPOCHS + 1), history['val_rmse'], label='Validation RMSE', marker='o')
plt.title('xLSTM Pure Sequence Training Curve')
plt.xlabel('Epochs')
plt.ylabel('RMSE')
plt.legend()
plt.grid(True)
plt.savefig('xlstm_training_curve.png')
plt.show()
    
print(f"完成訓練 Best Val RMSE: {best_val_rmse:.4f}")

gc.collect()
torch.cuda.empty_cache()

# #提交Kaggle競賽時inference用
# model_path = "/kaggle/input/pure-xlstm-model/xlstm_model.pth"
# print(f"正載入權重{model_path}...")
# state_dict = torch.load(model_path, map_location=device)
# model.load_state_dict(state_dict)

In [ ]:
#特徵提取(TxLSTM Embeddings)
print("正在提取xLSTM Embeddings")
model.eval()
xlstm_feats_list = []
train_loader_seq = DataLoader(dataset, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=0)
with torch.no_grad():
    for batch in tqdm(train_loader_seq):
        cat = batch['cat_features'].to(device)
        num = batch['num_features'].to(device)
        embeddings = model.extract_features(cat, num).cpu().numpy()
        xlstm_feats_list.append(embeddings)

xlstm_df = pd.DataFrame(
    np.vstack(xlstm_feats_list), 
    columns=[f'lstm_emb_{i}' for i in range(cfg.EMBED_DIM * 2)]
)

xlstm_df['id'] = dataset.essay_ids

full_train_df = train_handcrafted.merge(xlstm_df, on='id', how='left')
X = full_train_df.drop(['id', 'score'], axis=1)
y = full_train_df['score']

#清理記憶體
del train_logs, dataset, dataloader, train_loader_seq
gc.collect()


In [ ]:
import seaborn as sns
import pandas as pd
from sklearn.model_selection import StratifiedKFold
#訓練 XGBoost
print("訓練XGBoost")

#製作分層標籤
y_strata = pd.cut(y, bins=15, labels=False)

#初始化 StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

xgb_models = []
xgb_history = []
oof_preds = np.zeros(len(X))

xgb_params = {
    'n_estimators': 2000,
    'learning_rate': 0.01,
    'max_depth': 6,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'objective': 'reg:squarederror',
    'n_jobs': -1,
    'tree_method': 'hist',
    'device': 'cuda',
    'random_state': 42,
    'eval_metric': 'rmse'
}

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_strata)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    xgb_model = xgb.XGBRegressor(**xgb_params)
    
    xgb_model.fit(
        X_train, y_train,
        eval_set=[(X_train, y_train), (X_val, y_val)], 
        early_stopping_rounds=100,
        verbose=False
    )
    
    results = xgb_model.evals_result()
    xgb_history.append(results)

    val_preds = xgb_model.predict(X_val)
    oof_preds[val_idx] = val_preds
    xgb_models.append(xgb_model)
    
    rmse = mean_squared_error(y_val, val_preds, squared=False)
    print(f"Fold {fold+1} RMSE: {rmse:.4f}")

print(f"Overall XGBoost OOF RMSE: {mean_squared_error(y, oof_preds, squared=False):.4f}")

del full_train_df, xlstm_df, train_handcrafted, oof_preds, xgb_history
gc.collect()
torch.cuda.empty_cache()


#Inference
def generate_xgb_submission(xgb_models, xlstm_model, tfidf_vec, encoders, scaler, test_logs_path, output_path='submission.csv'):
    print(f"正載入測試資料 {test_logs_path}...")

    test_logs = pd.read_csv(test_logs_path)
    test_logs = preprocess_logs(test_logs)

    test_logs, _, _, _, _, _ = encode_and_scale(test_logs, encoders, scaler)

    test_handcrafted, _ = make_handcrafted_features(test_logs, tfidf_vectorizer=tfidf_vec, is_train=False)

    test_dataset = KeystrokeDataset(test_logs, {}, max_len=cfg.MAX_SEQ_LEN)

    test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False, num_workers=0)
    
    xlstm_model.eval()
    test_emb_list = []
    
    with torch.no_grad():
        for batch in tqdm(test_loader):
            cat = batch['cat_features'].to(device)
            num = batch['num_features'].to(device)

            emb = xlstm_model.extract_features(cat, num).cpu().numpy()
            test_emb_list.append(emb)

    test_xlstm_df = pd.DataFrame(
    np.vstack(test_emb_list), 
    columns=[f'lstm_emb_{i}' for i in range(cfg.EMBED_DIM * 2)]
    )
    test_xlstm_df['id'] = test_dataset.essay_ids 

    full_test_df = test_handcrafted.merge(test_xlstm_df, on='id', how='left')
    X_test = full_test_df.drop(['id'], axis=1, errors='ignore')

    for col in X.columns:
        if col not in X_test.columns:
            X_test[col] = 0
    X_test = X_test[X.columns]

    final_preds = np.zeros(len(X_test))
    for model in xgb_models:
        final_preds += model.predict(X_test) / len(xgb_models)
        
    final_preds = np.clip(final_preds, 0.5, 6.0)
    submission = pd.DataFrame({'id': full_test_df['id'], 'score': final_preds})
    submission.to_csv(output_path, index=False)
    print(f"XGBoost Submission 保存 {output_path}")

gc.collect()
torch.cuda.empty_cache()

#執行inference
generate_xgb_submission(xgb_models, model, tfidf_vec, encoders, scaler, cfg.TEST_LOGS_PATH)

In [ ]:
#XGBoost平均訓練曲線圖
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

train_rmse_list = []
val_rmse_list = []

min_len = min([len(h['validation_0']['rmse']) for h in xgb_history])

for h in xgb_history:
    train_rmse_list.append(h['validation_0']['rmse'][:min_len])
    val_rmse_list.append(h['validation_1']['rmse'][:min_len])

train_rmse_arr = np.array(train_rmse_list)
val_rmse_arr = np.array(val_rmse_list)

train_mean = np.mean(train_rmse_arr, axis=0)
train_std = np.std(train_rmse_arr, axis=0)
val_mean = np.mean(val_rmse_arr, axis=0)
val_std = np.std(val_rmse_arr, axis=0)

x_axis = range(len(train_mean))

sns.set_style("whitegrid")
plt.figure(figsize=(10, 6))

plt.plot(x_axis, train_mean, label='Average Train RMSE', color='blue')
plt.fill_between(x_axis, train_mean - train_std, train_mean + train_std, color='blue', alpha=0.15)

plt.plot(x_axis, val_mean, label='Average Validation RMSE', color='red')
plt.fill_between(x_axis, val_mean - val_std, val_mean + val_std, color='red', alpha=0.15)

best_idx = np.argmin(val_mean)
best_score = val_mean[best_idx]

plt.annotate(f'Best Avg Val: {best_score:.4f}', 
             xy=(best_idx, best_score), 
             xytext=(best_idx, best_score + 0.1),
             arrowprops=dict(facecolor='black', shrink=0.05),
             fontsize=12)

plt.title('XGBoost Learning Curve (Average over 5 Folds)', fontsize=16, fontweight='bold')
plt.xlabel('Boosting Rounds', fontsize=14)
plt.ylabel('RMSE Score', fontsize=14)
plt.legend(fontsize=12)

plt.tight_layout()
plt.savefig('average_learning_curve.png', dpi=300)
plt.show()

In [ ]:
#XGBoost特徵重要性圖
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

feature_scores = {}

for model in xgb_models:
    scores = model.get_booster().get_score(importance_type='weight')
    for feature, score in scores.items():
        if feature not in feature_scores:
            feature_scores[feature] = []
        feature_scores[feature].append(score)

avg_scores = []
for feature, score_list in feature_scores.items():
    avg_score = sum(score_list) / len(xgb_models) 
    avg_scores.append({'Feature': feature, 'Average F-Score': avg_score})

df_importance = pd.DataFrame(avg_scores)
df_top50 = df_importance.sort_values(by='Average F-Score', ascending=False).head(50)

plt.figure(figsize=(10, 8))
sns.set_style("whitegrid")

ax = sns.barplot(data=df_top50, x='Average F-Score', y='Feature', color='#1f77b4')
for i, v in enumerate(df_top50['Average F-Score']):
    ax.text(v + 1, i + 0.25, str(round(v, 1)), color='black', fontsize=10)

plt.title('Top 50 Feature Importance (Average over 5 Folds)', fontsize=16, fontweight='bold')
plt.xlabel('Average F-Score', fontsize=14)
plt.ylabel('Features', fontsize=14)
plt.tight_layout()

plt.savefig('average_feature_importance.png', dpi=300)
plt.show()